In [2]:
import pandas as pd

fund_master = pd.read_csv("../data/raw/fund_master.csv")
nav_history = pd.read_csv("../data/raw/nav_history.csv")
aum_by_fund_house = pd.read_csv("../data/raw/aum_by_fund_house.csv")
monthly_sip_inflows = pd.read_csv("../data/raw/monthly_sip_inflows.csv")
category_inflows = pd.read_csv("../data/raw/category_inflows.csv")
industry_folio_count = pd.read_csv("../data/raw/industry_folio_count.csv")
scheme_performance = pd.read_csv("../data/raw/scheme_performance.csv")
investor_transactions = pd.read_csv("../data/raw/investor_transactions.csv")
portfolio_holdings = pd.read_csv("../data/raw/portfolio_holdings.csv")
benchmark_indices = pd.read_csv("../data/raw/benchmark_indices.csv")

In [3]:
# 1. fund_master.csv

In [4]:
fund_master["launch_date"] = pd.to_datetime(fund_master["launch_date"])

In [5]:
print(fund_master.dtypes)

amfi_code                      int64
fund_house                    object
scheme_name                   object
category                      object
sub_category                  object
plan                          object
launch_date           datetime64[ns]
benchmark                     object
expense_ratio_pct            float64
exit_load_pct                float64
min_sip_amount                 int64
min_lumpsum_amount             int64
fund_manager                  object
risk_category                 object
sebi_category_code            object
dtype: object


In [6]:
numeric_cols = ['expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount']
for col in numeric_cols:
    fund_master[col] = pd.to_numeric(fund_master[col], errors='coerce')

In [7]:
# 2. nav_history.csv

In [8]:
nav_history['date'] = pd.to_datetime(nav_history['date'])

In [9]:
print(nav_history.dtypes)

amfi_code             int64
date         datetime64[ns]
nav                 float64
dtype: object


In [10]:
nav_history['nav'] = pd.to_numeric(nav_history['nav'], errors='coerce')

In [11]:
nav_history = nav_history.sort_values(['amfi_code', 'date']).reset_index(drop=True) #Sorting

In [12]:
nav_history.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [13]:
nav_history = nav_history.drop_duplicates(subset=['amfi_code', 'date'], keep='first')

In [14]:
nav_history['nav'] = nav_history.groupby('amfi_code')['nav'].ffill() #forward fill

In [15]:
invalid_nav = nav_history[nav_history['nav'] <= 0] #validation
if len(invalid_nav) > 0:
    print(len(invalid_nav))
else:
    print("No invalid NAV")

No invalid NAV


In [16]:
# 3. aum_by_fund_house.csv

In [17]:
aum_by_fund_house['date'] = pd.to_datetime(aum_by_fund_house['date'])

In [18]:
print(aum_by_fund_house.dtypes)

date              datetime64[ns]
fund_house                object
aum_lakh_crore           float64
aum_crore                  int64
num_schemes                int64
dtype: object


In [19]:
numeric_cols = ['aum_lakh_crore', 'aum_crore', 'num_schemes']
for col in numeric_cols:
    aum_by_fund_house[col] = pd.to_numeric(aum_by_fund_house[col], errors='coerce')

In [20]:
# 4. monthly_sip_inflows.csv

In [21]:
monthly_sip_inflows['month'] = pd.to_datetime(monthly_sip_inflows['month'] + '-01')

In [22]:
print(monthly_sip_inflows.dtypes)

month                        datetime64[ns]
sip_inflow_crore                      int64
active_sip_accounts_crore           float64
new_sip_accounts_lakh               float64
sip_aum_lakh_crore                  float64
yoy_growth_pct                      float64
dtype: object


In [23]:
numeric_cols = ['sip_inflow_crore', 'active_sip_accounts_crore', 'new_sip_accounts_lakh', 'sip_aum_lakh_crore', 'yoy_growth_pct']
for col in numeric_cols:
    monthly_sip_inflows[col] = pd.to_numeric(monthly_sip_inflows[col], errors='coerce')

In [24]:
monthly_sip_inflows = monthly_sip_inflows.sort_values('month').reset_index(drop=True) #Sorting

In [25]:
monthly_sip_inflows.head()

,month,sip_inflow_crore,active_sip_accounts_crore,new_sip_accounts_lakh,sip_aum_lakh_crore,yoy_growth_pct
0,2022-01-01,11517,4.91,9.10,4.80,NaN
1,2022-02-01,11438,4.93,8.20,4.85,NaN
2,2022-03-01,12328,5.09,10.50,5.01,NaN
3,2022-04-01,11863,5.48,9.52,5.12,NaN
4,2022-05-01,12286,5.55,8.10,5.15,NaN


In [26]:
monthly_sip_inflows.isnull().sum()

month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

In [27]:
# Observation:
# yoy_growth_pct column contains missing values for 2022 months.
# These are expected because year-over-year growth requires previous year data, which is unavailable.
# No imputation was performed to avoid creating artificial growth values.

In [28]:
# 5. category_inflows.csv

In [29]:
category_inflows.head()

,month,category,net_inflow_crore
0,2024-04,Large Cap,2413.0
1,2024-04,Mid Cap,3897.0
2,2024-04,Small Cap,3533.0
3,2024-04,Flexi Cap,4947.0
4,2024-04,Large & Mid Cap,4214.0


In [30]:
category_inflows['month'] = pd.to_datetime(category_inflows['month'] + '-01')

In [31]:
category_inflows.head()

,month,category,net_inflow_crore
0,2024-04-01,Large Cap,2413.0
1,2024-04-01,Mid Cap,3897.0
2,2024-04-01,Small Cap,3533.0
3,2024-04-01,Flexi Cap,4947.0
4,2024-04-01,Large & Mid Cap,4214.0


In [32]:
category_inflows['net_inflow_crore'] = pd.to_numeric(category_inflows['net_inflow_crore'], errors='coerce')

In [33]:
category_inflows = category_inflows.sort_values(['month', 'category']).reset_index(drop=True) #Sorting

In [34]:
category_inflows.head()

,month,category,net_inflow_crore
0,2024-04-01,ELSS,466.0
1,2024-04-01,Flexi Cap,4947.0
2,2024-04-01,Gilt,784.0
3,2024-04-01,Hybrid,2955.0
4,2024-04-01,Large & Mid Cap,4214.0


In [35]:
# 6. industry_folio_count.csv

In [36]:
print(industry_folio_count.dtypes)

month                   object
total_folios_crore     float64
equity_folios_crore    float64
debt_folios_crore      float64
hybrid_folios_crore    float64
others_folios_crore    float64
dtype: object


In [37]:
industry_folio_count['month'] = pd.to_datetime(industry_folio_count['month'] + '-01', format='%Y-%m-%d')

In [38]:
print(industry_folio_count.dtypes)

month                  datetime64[ns]
total_folios_crore            float64
equity_folios_crore           float64
debt_folios_crore             float64
hybrid_folios_crore           float64
others_folios_crore           float64
dtype: object


In [39]:
numeric_cols = ['total_folios_crore', 'equity_folios_crore', 'debt_folios_crore', 'hybrid_folios_crore', 'others_folios_crore']
for col in numeric_cols:
    industry_folio_count[col] = pd.to_numeric(industry_folio_count[col], errors='coerce')

In [40]:
industry_folio_count = industry_folio_count.sort_values('month').reset_index(drop=True)

In [41]:
industry_folio_count.head()

,month,total_folios_crore,equity_folios_crore,debt_folios_crore,hybrid_folios_crore,others_folios_crore
0,2022-01-01,13.26,9.28,1.86,0.80,1.33
1,2022-04-01,13.91,9.74,1.95,0.83,1.39
2,2022-07-01,13.85,9.69,1.94,0.83,1.38
3,2022-10-01,14.12,9.88,1.98,0.85,1.41
4,2023-01-01,14.81,10.37,2.07,0.89,1.48


In [42]:
# 7. scheme_performance.csv

In [43]:
print(scheme_performance.dtypes)

amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object


In [44]:
scheme_performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [45]:
print("\nExpense Ratio Summary:")
print(scheme_performance['expense_ratio_pct'].describe())


Expense Ratio Summary:
count    40.000000
mean      1.237000
std       0.386584
min       0.550000
25%       0.787500
50%       1.425000
75%       1.540000
max       1.640000
Name: expense_ratio_pct, dtype: float64


In [46]:
invalid_expense = scheme_performance[
    (scheme_performance["expense_ratio_pct"] < 0.1) | (scheme_performance["expense_ratio_pct"] > 2.5)
]
print("Invalid  rows:", len(invalid_expense))

Invalid  rows: 0


In [47]:
# 8. investor_transactions.csv
investor_transactions['transaction_date'] = pd.to_datetime(investor_transactions['transaction_date'])

In [48]:
print(investor_transactions.dtypes)

investor_id                   object
transaction_date      datetime64[ns]
amfi_code                      int64
transaction_type              object
amount_inr                     int64
state                         object
city                          object
city_tier                     object
age_group                     object
gender                        object
annual_income_lakh           float64
payment_mode                  object
kyc_status                    object
dtype: object


In [49]:
invalid_amount = investor_transactions[
    investor_transactions["amount_inr"] <= 0
]

if len(invalid_amount) == 0:
    print("Valid amounts")
else:
    print("Invalid amount present")

Valid amounts


In [50]:
# 9. portfolio_holdings.csv

In [51]:
portfolio_holdings['portfolio_date'] = pd.to_datetime(portfolio_holdings['portfolio_date'])

In [52]:
print(portfolio_holdings.dtypes)

amfi_code                     int64
stock_symbol                 object
stock_name                   object
sector                       object
weight_pct                  float64
market_value_cr             float64
current_price_inr           float64
portfolio_date       datetime64[ns]
dtype: object


In [53]:
portfolio_holdings.head()

,amfi_code,stock_symbol,stock_name,sector,weight_pct,market_value_cr,current_price_inr,portfolio_date
0,119551,POWERGRID,Power Grid Corporation,Utilities,13.85,737.09,6011.08,2025-12-31
1,119551,HDFCBANK,HDFC Bank Ltd,Banking,11.19,88.97,1074.65,2025-12-31
2,119551,GRASIM,Grasim Industries Ltd,Diversified,9.90,208.45,5964.59,2025-12-31
3,119551,DRREDDY,Dr. Reddy's Laboratories,Pharma,4.76,161.32,3748.82,2025-12-31
4,119551,ASIANPAINT,Asian Paints Ltd,Paints,10.25,725.90,1321.45,2025-12-31


In [54]:
# 10. benchmark_indices.csv
print(benchmark_indices.dtypes)

date            object
index_name      object
close_value    float64
dtype: object


In [55]:
benchmark_indices['date'] = pd.to_datetime(benchmark_indices['date'])

In [56]:
benchmark_indices.head()

,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15
